In [36]:
import pandas as pd
from pyrosetta import init, pose_from_pdb
from pathlib import Path
from itertools import chain

In [2]:
init(options=[
    '-use_input_sc',
    '-input_ab_scheme', 'AHo_Scheme',
    '-ignore_unrecognized_res',
    '-ignore_zero_occupancy', 'false',
    '-load_PDB_components', 'false',
    '-relax:default_repeats', '2',
    '-no_fconfig',
    '-mute', 'all'  
])

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python313.Release 2026.28+release.188eabbe2c00638eb408e86cbd67b1fec355b9c7 2026-07-09T14:07:17] retrieved from: http://www.pyrosetta.org


In [3]:
def get_sequence(pdb_file):
    pose = pose_from_pdb(pdb_file)
    return pose.sequence()


In [4]:
get_sequence('pdb_files/7K18_AQPMSSSPKET_mutant.pdb')

'DDQSPEKVNILAKINLLFVAIFTGECIVKMAALRHYYFTNSWNIFDFVVVILSIVAQPMSSSPKETFFSPTLFRVIRLARIGRILRLIRGAKGIRTLLFAVRDGYIAQPENCVYHCFPGSSGCDTLCKEKGGTSGHCGFKVGHGLACWCNALPDNVGIIVEGEKCHS'

In [9]:
def compare_sequence(original, new):
    seq = get_sequence(original)
    new_seq = get_sequence(new)
    mutations = []

    for num, (i, j) in enumerate(zip(seq, new_seq)):
        if i != j:
            mutations.append(f'{i}{num}{j}')
    return mutations

                

In [ ]:
compare_sequence('pdb_files/7K18_AQPMSSSPKET_mutant.pdb', 'variants/dg_baseline/DDG_6.pdb')

['A106I',
 'Q107Y',
 'P108Y',
 'E109I',
 'V112Y',
 'Y113V',
 'P117S',
 'S119H',
 'S120M',
 'V140W',
 'G141I',
 'G156C',
 'I158W',
 'V159A',
 'E160M',
 'G161M',
 'H165L']

In [39]:
def grab_pdbs(function):
    parent_dir = Path('variants')

    # Finds all .pdb files inside dg* folders
    pdb_files = list(parent_dir.glob(f"{function}*/**/*.pdb"))
    return pdb_files

In [42]:

def mutation_df(pdbs):
    data = []
    paths = []

    for pdb in pdbs:
        d = compare_sequence('pdb_files/7K18_AQPMSSSPKET_mutant.pdb', str(pdb))
        data.append(d)
        path = [str(pdb) for i in range(len(d))]
        paths.append(path)



    df = pd.DataFrame({
        'path': list(chain.from_iterable(paths)),
        'mutation': list(chain.from_iterable(data))
    })

    return df

In [ ]:
pdbs = grab_pdbs('dg')
df = mutation_df(pdbs)
top_mutations = df['mutation'].value_counts().head(10)
print(top_mutations)

In [ ]:
pdbs = grab_pdbs('distance')
df = mutation_df(pdbs)
top_mutations = df['mutation'].value_counts().head(10)
print(top_mutations)